In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

# .envファイルからAPIキーを読み込む
load_dotenv()
BASE = "https://api.planet.com/data/v1"
API_KEY = os.environ.get("PL_API_KEY") 
if not API_KEY:
    raise RuntimeError('PL_API_KEY is not set in .env')
AUTH = (API_KEY, "")


In [ ]:
def quick_search(item_type="PSScene",
                 date_gte="2025-01-24T00:00:00Z",
                 date_lte="2025-01-25T23:59:59Z",
                 cloud_lte=0.1,
                 bbox=(-106.90, 34.90, -106.40, 35.30),  # (minLon, minLat, maxLon, maxLat) around Albuquerque
                 limit=5):
    geom = {
        "type": "Polygon",
        "coordinates": [[
            [bbox[0], bbox[1]],
            [bbox[2], bbox[1]],
            [bbox[2], bbox[3]],
            [bbox[0], bbox[3]],
            [bbox[0], bbox[1]],
        ]]
    }

    payload = {
        "item_types": [item_type],
        "filter": {
            "type": "AndFilter",
            "config": [
                {"type": "DateRangeFilter", "field_name": "acquired",
                 "config": {"gte": date_gte, "lte": date_lte}},
                {"type": "RangeFilter", "field_name": "cloud_cover",
                 "config": {"lte": cloud_lte}},
                {"type": "GeometryFilter", "field_name": "geometry",
                 "config": geom},
            ],
        },
    }

    r = requests.post(f"{BASE}/quick-search", auth=AUTH, json=payload)
    r.raise_for_status()
    return r.json().get("features", [])[:limit]

def get_assets(item_type, item_id):
    r = requests.get(f"{BASE}/item-types/{item_type}/items/{item_id}/assets", auth=AUTH)
    r.raise_for_status()
    return r.json()

def activate(item_type, item_id, asset_type):
    assets = get_assets(item_type, item_id)
    a = assets.get(asset_type)
    if not a:
        raise KeyError(f"Asset '{asset_type}' not available. Available: {[k for k in assets if not k.startswith('_')]}")
    act = a.get("_links", {}).get("activate")
    if not act:
        raise KeyError(f"No activate link for '{asset_type}' (permission or availability issue).")
    r = requests.post(act, auth=AUTH)
    r.raise_for_status()

def ensure_active_and_get_location(item_type, item_id, asset_type, timeout=900, poll=10):
    t0 = time.time()
    activated = False

    while True:
        assets = get_assets(item_type, item_id)
        a = assets.get(asset_type, {})
        status = a.get("status")
        loc = a.get("location")  # <-- IMPORTANT: top-level "location"
        msg = a.get("message") or a.get("error") or ""

        # activate only once, only if needed
        if (not activated) and status == "inactive":
            activate(item_type, item_id, asset_type)
            activated = True

        print(f"{asset_type}: status={status} {msg}".strip())

        if status == "active" and loc:
            return loc

        if status == "failed":
            raise RuntimeError(f"{asset_type} failed: {a}")

        if time.time() - t0 > timeout:
            raise TimeoutError(f"Timed out waiting for {asset_type} (last status={status})")

        time.sleep(poll)

def download(url, out_path):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


In [ ]:
item_type = "PSScene"

    # 1) find one scene
feats = quick_search(item_type=item_type, cloud_lte=0.1, limit=1)
if not feats:
    raise RuntimeError("No scenes found. Try widening date range or cloud_lte (e.g., 0.2).")

item_id = feats[0]["id"]
cc = feats[0]["properties"].get("cloud_cover")
acquired = feats[0]["properties"].get("acquired")
print("item_id:", item_id, "cloud_cover:", cc, "acquired:", acquired)

In [ ]:
assets = get_assets(item_type, item_id)   # dict
print(sorted(assets.keys()))


In [ ]:
for asset_type in ["basic_analytic_4b", "basic_analytic_4b_xml", "basic_udm2"]:
    loc = ensure_active_and_get_location(item_type, item_id, asset_type, timeout=1800, poll=10)

    ext = ".xml" if "xml" in asset_type else ".tif"
    out = f"../data/{item_id}_{asset_type}{ext}"

    download(loc, out)
    print("downloaded:", out)


In [ ]:
for asset_type in ["ortho_analytic_4b_sr"]:
    loc = ensure_active_and_get_location(item_type, item_id, asset_type, timeout=1800, poll=10)

    ext = ".xml" if "xml" in asset_type else ".tif"
    out = f"../data/{item_id}_{asset_type}{ext}"

    download(loc, out)
    print("downloaded:", out)